# 01 - Preparação & Inflate do Dataset (Dogs vs. Cats)

**Objetivo:** organizar as imagens do *Dogs vs. Cats* em `/tf/data/raw_jpg/` e ajustar o tamanho do
diretório para **~1,5 GB** (acima do mínimo de **1 GB** exigido na disciplina), via cópia iterativa.

**Por quê ~1,5 GB (e não 6-8 GB como antes)?** Os testes do notebook 02 mostraram que o gargalo da
ingestão é o **decode JPEG na CPU**, não o disco — e o decode roda a cada época independente de cache
(medimos *cold* == *warm*: 1.143 vs 1.134 img/s). Logo **não precisamos de um dataset maior que a RAM**
pra expor o problema: basta cumprir o mínimo de 1 GB e ter imagens suficientes pra rodadas estáveis.
Dataset menor também é **reprodutível** — ninguém precisa gerar dezenas de GB pra testar o trabalho.

> Roda **dentro do contêiner**. Os caminhos usam `/tf/data` (volume mapeado de `./data`). — confira o
> Dockerfile e o docker-compose pra mais detalhes.

## Pré-requisito: download manual do dataset

O download é **manual** — não usamos a API do Kaggle. Passo a passo:

1. Acesse <https://www.kaggle.com/competitions/dogs-vs-cats/data> logado e **aceite as regras** da
   competição (botão *Join Competition* / *I Understand and Accept*), detalhe: pra conseguir se juntar competição será necessário ter um número de telefone associado a sua conta.
2. Baixe o **`train.zip`** (~543 MB, 25.000 imagens com a nomenclatura: `cat.0.jpg`, `dog.0.jpg`...).
3. Coloque o `train.zip` em `./data/raw_jpg/` (no host) e descompacte:
   ```bash
   cd data/raw_jpg && unzip train.zip && rm train.zip
   ```
   Isso cria `./data/raw_jpg/train/` com as 25.000 imagens.

A partir daqui, as células abaixo cuidam de mover de `train/` para `raw_jpg/` e de inflar o dataset.

In [1]:
import shutil
import time
from pathlib import Path

# Configuração de caminhos e da meta de tamanho
BASE_DATA = Path("/tf/data")          # volume mapeado de ./data
RAW_DIR   = Path("/tf/data/raw_jpg")      # destino final das imagens .jpg
TRAIN_DIR = Path("/tf/data/raw_jpg/train")       # subpasta criada pelo unzip do train.zip

TARGET_GB = 1.5                        # meta de tamanho (>= 1 GB exigido pelo professor)

RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f"RAW_DIR = {RAW_DIR}")
print(f"Meta    = {TARGET_GB} GB")

RAW_DIR = /tf/data/raw_jpg
Meta    = 1.5 GB


In [2]:
# Mover as imagens com label(comeca com cat ou dog) de raw_jpg/train/ para raw_jpg/
# Se ja estiverem direto em RAW_DIR, so reporta e segue.
if TRAIN_DIR.exists():
    labeled = []
    for img in TRAIN_DIR.rglob("*.jpg"): # Procura na pasta e subpastas pela extecao .jpg, confirma se o nome comeca com cat ou dog e adiciona na lista
        if img.name.lower().startswith(("cat", "dog")):
            labeled.append(img)
    print(f"Encontradas {len(labeled)} imagens com label no dir {TRAIN_DIR}")

    moved = 0
    for img in labeled: #Pra cada img dentro da lista, move para a pasta RAW_DIR, se ainda nao existir la e add +1 no contador
        destiny = RAW_DIR / img.name
        if not destiny.exists():
            shutil.move(img, destiny)
            moved += 1
    print(f"{moved} imagens movidas para {RAW_DIR}")

    shutil.rmtree(TRAIN_DIR, ignore_errors=True) # Apaga a pasta train/
    print(f"Subpasta {TRAIN_DIR.name}/ removida.")

else: # Confere se as imagens com label ja estao na pasta RAW_DIR, se nao estiverem, levanta um erro 
    base = [p for p in RAW_DIR.glob("*.jpg") if p.name.lower().startswith(("cat", "dog"))]
    if base:
        print(f"Sem subpasta train/ — {len(base)} imagens já tão em {RAW_DIR}.")
    else:
        raise FileNotFoundError(
            f"Nenhuma imagem encontrada. Coloque o train.zip em {RAW_DIR} e descompacta"
            "(oia a célula de pre-requisito acima e olha o read me)."
        )

Encontradas 25000 imagens com label no dir /tf/data/raw_jpg/train
25000 imagens movidas para /tf/data/raw_jpg
Subpasta train/ removida.


## Inflate: duplicação iterativa até a meta de tamanho

A função abaixo varre as imagens e as copia, anexando um sufixo `__dupN` ao nome
**preservando o prefixo `cat`/`dog`** — assim os notebooks de benchmark continuam extraindo o label pelo
nome do arquivo. A cada N cópias mede o tamanho do diretório e para ao cruzar `TARGET_GB`.

In [3]:
def inflate_directory(path, target_gb):
    # Monta uma lista c todos os arquivos .jpg que nao sao duplicados (nao contem __dup no nome)
    base_files = [p for p in path.glob("*.jpg") if "__dup" not in p.name]
    if not base_files:
        raise RuntimeError("Nenhuma imagem-base encontrada. Rode a célula anterior.")

    # Calcula o tamanho atual do diretório e o tamanho alvo em bytes
    current_bytes = sum(f.stat().st_size for f in path.glob("*.jpg"))
    target_bytes  = target_gb * (1024 ** 3)

    print(f"Tamanho inicial: {current_bytes / (1024**3):.2f} GB")
    print(f"Base: {len(base_files):,} imagens")
    print(f"Meta: {target_gb} GB")

    # Se o tamanho atual já for maior ou igual ao alvo, retorna
    if current_bytes >= target_bytes:
        print("Meta atingida")
        return

    # Inicia os contadores de cópias e passes, e o tempo inicial
    copies, dup_pass = 0, 1
    t0 = time.time()

    # Loop principal: enquanto o tamanho atual for menor que o alvo, copia os arquivos base para novos arquivos com sufixo __dup{pass}
    while current_bytes < target_bytes:
        for arq in base_files:
            nome_final = path / f"{arq.stem}__dup{dup_pass}{arq.suffix}"
            if not nome_final.exists():
                shutil.copy(arq, nome_final)
                current_bytes += arq.stat().st_size  # Incrementa pelo tamanho do arquivo original, pq tem o msm tamanho e n precisa buscar no disco
                copies += 1
            if current_bytes >= target_bytes:
                break
        
        print(f"  Passe {dup_pass:02d} | {copies:,} cópias | "
              f"{current_bytes/(1024**3):.2f} GB | {time.time()-t0:.0f}s")
        dup_pass += 1

    print(f"\n✅ Inflate concluído: {current_bytes/(1024**3):.2f} GB "
          f"em {copies:,} cópias ({dup_pass-1} passes) — {time.time()-t0:.0f}s")


inflate_directory(RAW_DIR, TARGET_GB)

Tamanho inicial: 0.53 GB
Base: 25,000 imagens
Meta: 1.5 GB
  Passe 01 | 25,000 cópias | 1.07 GB | 1s
  Passe 02 | 45,433 cópias | 1.50 GB | 4s

✅ Inflate concluído: 1.50 GB em 45,433 cópias (2 passes) — 4s


In [4]:
# Verificando se deu tudo certo, reportando o total de arquivos e tamanho do diretório
files = list(RAW_DIR.glob("*.jpg"))
n_cat = sum(1 for f in files if f.name.lower().startswith("cat"))
n_dog = sum(1 for f in files if f.name.lower().startswith("dog"))
total_gb = sum(f.stat().st_size for f in files) / (1024 ** 3)

print(f"Dir: {RAW_DIR}")
print(f"Total de imagens: {len(files):,}")
print(f"Gatos : {n_cat:,} imgs")
print(f"Cachorros : {n_dog:,} imgs")
print(f"Tamanho: {total_gb:.2f} GB")
print("Exemplos de nomes:", [f.name for f in files[:3]] + [f.name for f in files[-3:]])

Dir: /tf/data/raw_jpg
Total de imagens: 70,433
Gatos : 35,207 imgs
Cachorros : 35,226 imgs
Tamanho: 1.50 GB
Exemplos de nomes: ['cat.5749.jpg', 'dog.4832.jpg', 'cat.4470__dup1.jpg', 'cat.9920__dup1.jpg', 'cat.8956__dup1.jpg', 'dog.3686.jpg']
